In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q langchain-groq

In [ ]:
!pip install -q tavily-python scrapegraph-py

In [ ]:
!pip install -q "crewai==0.86.0" "litellm==1.55.3"

#  Libraries & Tools

In [2]:
import os
import re
import asyncio
import time
import logging
import litellm
import warnings
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
os.environ["LITELLM_LOG"] = "ERROR"

from IPython.display import HTML, display
from kaggle_secrets import UserSecretsClient

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from langchain_groq import ChatGroq
from tavily import TavilyClient

In [22]:
# Remove the existing Crew execution lock to allow a new run
if os.path.exists(".crew_run_lock"):
    os.remove(".crew_run_lock")

In [23]:
# Solution to the cache_breakpoint problem
litellm.drop_params = True

# Disable prompt caching at the environment level
os.environ["LITELLM_PROMPT_CACHING"] = "false"

In [24]:
# Load the required API keys 
user_secrets = UserSecretsClient()

groq_api_key = user_secrets.get_secret("GROQ_API_KEY")
tavily_api_key = user_secrets.get_secret("TAVILY_API_KEY")

In [25]:
# Initialize the Groq language model
llm = LLM(
    model="groq/llama-3.3-70b-versatile",  
    api_key=groq_api_key,
    temperature=0
)

print("Groq Connected Successfully!")

Groq Connected Successfully!


In [26]:
client = TavilyClient(api_key=tavily_api_key)

In [27]:
# Configure limits and execution controls 
MAX_PRODUCTS = 2           
MAX_SCRAPER_URLS = 2       
TAVILY_MAX_RESULTS = 2 

USE_TAVILY_EXTRACT_FALLBACK = False
RUN_LOCK_FILE = ".crew_run_lock"

RUN_NOW = True

In [28]:
# Define Tools
class TavilySearchTool(BaseTool):
    name: str = "Tavily Search"
    description: str = (
        "Search the web for products and return product names, "
        "prices, specifications and URLs."
    )

    def _run(self, query: str) -> str:
        try:
            response = client.search(
                query=query,
                max_results=TAVILY_MAX_RESULTS
            )

            results = response.get("results", [])

            simplified_results = []

            for item in results:
                simplified_results.append({
                    "title": item.get("title"),
                    "url": item.get("url"),
                    "content": item.get("content")
                })

            return json.dumps(simplified_results, ensure_ascii=False, indent=2)

        except Exception as e:
            return f"Search error: {str(e)}"


search_tool = TavilySearchTool()

In [29]:
# Scraper tool
class ProductScraperTool(BaseTool):
    name: str = "Product Page Scraper"
    description: str = (
        "Scrapes product URLs. "
        "Input: comma-separated product URLs. "
        "Returns JSON with product name, price and specifications snippet."
    )

    def _run(self, urls: str) -> str:
        url_list = [
            url.strip()
            for url in str(urls).split(",")
            if url.strip().startswith("http")
        ]

        # Determine the number of links
        url_list = url_list[:MAX_SCRAPER_URLS]

        if not url_list:
            return "No valid URLs provided."

        results = []

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "Chrome/124.0 Safari/537.36"
            )
        }

        for url in url_list:
            item = {
                "url": url,
                "success": False
            }

            try:
                response = requests.get(
                    url,
                    headers=headers,
                    timeout=20
                )

                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, "html.parser")

                    # Product Name
                    title_tag = soup.find("meta", attrs={"property": "og:title"})

                    if title_tag and title_tag.get("content"):
                        item["name"] = title_tag["content"].strip()
                    elif soup.title and soup.title.string:
                        item["name"] = soup.title.string.strip()
                    else:
                        item["name"] = None

                    # Price
                    price_tag = soup.find("meta", attrs={"itemprop": "price"})

                    if not price_tag:
                        price_tag = soup.find(
                            "meta",
                            attrs={"property": "product:price:amount"}
                        )

                    if price_tag and price_tag.get("content"):
                        item["price"] = price_tag["content"].strip()
                    else:
                        item["price"] = None

                    # Remove unwanted tags
                    for tag in soup(["script", "style", "nav", "footer", "header"]):
                        tag.decompose()

                    page_text = soup.get_text(separator=" ", strip=True)
                    page_text = re.sub(r"\s+", " ", page_text)

                    item["specs_snippet"] = page_text[:2000]
                    item["success"] = True

                else:
                    item["error"] = f"HTTP status {response.status_code}"

            except Exception as e:
                item["error"] = str(e)

            # Fallback
            if USE_TAVILY_EXTRACT_FALLBACK and not item.get("success"):
                try:
                    extract_response = client.extract(urls=[url])

                    contents = []

                    for result in extract_response.get("results", []):
                        content = result.get("raw_content") or result.get("text") or ""
                        if content:
                            contents.append(content)

                    extracted_text = "\n".join(contents)

                    if extracted_text:
                        item["success"] = True
                        item["source"] = "tavily_extract"
                        item["specs_snippet"] = extracted_text[:2000]
                        item.pop("error", None)

                except Exception:
                    pass

            results.append(item)

        return json.dumps(results, ensure_ascii=False)[:8000]


scraper_tool = ProductScraperTool()

# Company Requirements

In [30]:
# Define the company requirements and constraints for the purchasing task
company_context = """
Company Name: ABC Technology

Industry: Software Development

Budget: $700 total.

Purpose:
Purchase smartphones for software engineers.

Requirements:
- Minimum 8GB RAM
- Minimum 256GB Storage
- Long battery life
- Good camera
- Reliable brand
- Best value for money

Important:
The selected smartphones must fit within the company budget.
"""

# Multi-Agent Procurement Workflow

In [31]:
# Search Agent
search_agent = Agent(
    role="Product Search Specialist",
    goal="Find the best products based on the company requirements.",
    backstory="""
    You are an expert procurement specialist.
    You search trusted online stores and collect accurate product information.
    """,
    llm=llm,
    tools=[search_tool],
    verbose=False,
    allow_delegation=False,
    max_iter=3
)

print("Search Agent Created Successfully!")

Search Agent Created Successfully!


In [32]:
# Scraper Agent
scraper_agent = Agent(
    role="Product Data Collection Specialist",
    goal="""
    Collect detailed product information from product websites.
    Extract accurate specifications and prices.
    """,
    backstory="""
    You are responsible for verifying product information
    and collecting structured data from websites.
    """,
    llm=llm,
    tools=[scraper_tool],
    verbose=False,
    allow_delegation=False,
    max_iter=3
)

print("Scraper Agent Created Successfully!")

Scraper Agent Created Successfully!


In [33]:
# Analysis Agent
analysis_agent = Agent(
    role="Procurement Analysis Specialist",
    goal="""
    Compare products and rank them based on
    company requirements, price, specifications,
    and overall value.
    """,
    backstory="""
    You are an expert procurement analyst.
    You evaluate products objectively and recommend the best options.
    """,
    llm=llm,
    verbose=False,
    allow_delegation=False,
    max_iter=3
)

print("Analysis Agent Created Successfully!")

Analysis Agent Created Successfully!


In [34]:
# Report Agent
report_agent = Agent(
    role="Procurement Report Writer",
    goal="""
    Generate a professional HTML procurement report.
    """,
    backstory="""
    You create business-ready procurement reports
    with clear recommendations.
    """,
    llm=llm,
    verbose=False,
    allow_delegation=False,
    max_iter=3
)

print("Report Agent Created Successfully!")

Report Agent Created Successfully!


In [35]:
# Define the task for searching smartphones that match the company requirements
search_task = Task(
    description=f"""
    Company Context:
    {company_context}

    Search the web using Tavily Search Tool.

    Find the best {MAX_PRODUCTS} smartphones under the company budget.

    For each product provide:
    - Product Name
    - Price
    - Key Specifications
    - Website URL

    Important:
    - Prefer direct product pages.
    - Use only information retrieved from the search tool.
    - Be concise.
    """,

    expected_output="""
    A short markdown table containing:
    - Product Name
    - Price
    - Key Specifications
    - Website URL
    """,

    agent=search_agent
)

In [36]:
# Define the task for extracting detailed specifications from the identified products
scraping_task = Task(
    description=f"""
    Use the Product Page Scraper tool to collect detailed information
    from at most {MAX_SCRAPER_URLS} product URLs returned by the previous task.

    If there are multiple URLs, pass them together as one comma-separated list.

    Extract:
    - Product Name
    - Current Price
    - RAM
    - Storage
    - Processor
    - Battery
    - Camera
    - Display
    - Product URL

    If any field is missing, write "Not found".

    Be concise and return structured data.
    """,

    expected_output="""
    A structured list or table containing detailed specifications
    for each smartphone.
    """,

    agent=scraper_agent,
    context=[search_task]
)

In [37]:
# Define the task for evaluating and ranking the smartphones based on the company requirements
analysis_task = Task(
    description=f"""
    Analyze the collected product information from the previous task.

    Company Requirements:
    {company_context}

    Compare products based on:
    - Price
    - RAM
    - Storage
    - Battery
    - Camera
    - Value for money

    Rank the products from best to worst.

    Be concise.
    """,

    expected_output="""
    A short ranked comparison table with:
    - Rank
    - Product Name
    - Score
    - Strengths
    - Weaknesses
    - Final Recommendation
    """,

    agent=analysis_agent,
    context=[scraping_task]
)

In [38]:
# Define the task for generating the final HTML procurement report
report_task = Task(
    description="""
    Create a professional procurement report using the analysis results
    from the previous task.

    The report should include:

    1. Executive Summary
    2. Company Requirements
    3. Product Comparison
    4. Recommended Product
    5. Reasons for Selection

    Important:
    - Output raw valid HTML only.
    - Do NOT wrap the HTML inside markdown code fences.
    - Do NOT add explanations outside the HTML.
    - Use simple inline CSS.
    - Be concise.
    """,

    expected_output="""
    A complete HTML procurement report.
    """,

    agent=report_agent,
    context=[analysis_task],
    output_file="procurement_report.html"
)

# Execution & Results

In [39]:
# Crew Execution
final_crew = Crew(
    agents=[search_agent,scraper_agent,analysis_agent,report_agent],
    tasks=[search_task,scraping_task,analysis_task,report_task],

    process=Process.sequential,
    verbose=False,
    cache=False
)

In [40]:
# Preview the CrewAI agents and tasks without making API calls
def preview_crew_workflow():
    print("No Groq or Tavily API calls should happen from this preview.")
    print("=" * 60)

    print("\nAgents:")
    for agent in final_crew.agents:
        print(f"- {agent.role}")

    print("\nTasks:")

    for i, task in enumerate(final_crew.tasks, start=1):
        agent_role = task.agent.role if task.agent else "Unknown Agent"

        print("\n" + "=" * 60)
        print(f"Task {i} | Agent: {agent_role}")
        print("=" * 60)

        print("Description:")
        print(task.description[:700])

        print("\nExpected Output:")
        print(task.expected_output[:300])

    print("\n" + "=" * 60)
    print("Preview finished.")
    print("=" * 60)

preview_crew_workflow()

No Groq or Tavily API calls should happen from this preview.

Agents:
- Product Search Specialist
- Product Data Collection Specialist
- Procurement Analysis Specialist
- Procurement Report Writer

Tasks:

Task 1 | Agent: Product Search Specialist
Description:

    Company Context:
    
Company Name: ABC Technology

Industry: Software Development

Budget: $700 total.

Purpose:
Purchase smartphones for software engineers.

Requirements:
- Minimum 8GB RAM
- Minimum 256GB Storage
- Long battery life
- Good camera
- Reliable brand
- Best value for money

Important:
The selected smartphones must fit within the company budget.


    Search the web using Tavily Search Tool.

    Find the best 2 smartphones under the company budget.

    For each product provide:
    - Product Name
    - Price
    - Key Specifications
    - Website URL

    Important:
    - Prefer direct product pages.
    - Use only information retrieved from the search tool.
    - Be co

Expected Output:

    A short markdow

In [41]:
# Safe operation cell once
async def run_crew_once():
    if os.path.exists(RUN_LOCK_FILE):
        print("The project has already been operational once.")
        print(f"os.remove('{RUN_LOCK_FILE}')")
        return None

    max_retries = 3
    retry_delay = 10  

    for attempt in range(max_retries):
        try:
            with open(RUN_LOCK_FILE, "w", encoding="utf-8") as f:
                f.write("running")
            
            print(f"Starting Crew run (attempt {attempt + 1}/{max_retries})...")
            output = await final_crew.kickoff_async()
            
            with open(RUN_LOCK_FILE, "w", encoding="utf-8") as f:
                f.write("finished")
            
            print("Crew finished successfully.")
            return output
            
        except Exception as e:
            error_msg = str(e)
            
            if "RateLimitError" in error_msg or "rate_limit_exceeded" in error_msg:
                print(f"Rate limit hit. Waiting {retry_delay} seconds before retry...")
                if attempt < max_retries - 1:
                    await asyncio.sleep(retry_delay)
                    continue
                else:
                    print("Max retries reached. Please wait a few minutes and try again.")
                    raise
            else:
                print("Crew run failed.")
                raise
                
if RUN_NOW:
    result = await run_crew_once()
else:
    result = None
    print("Safe Mode: RUN_NOW = False")

Starting Crew run (attempt 1/3)...
Crew finished successfully.


In [42]:
# Display an HTML report after operation
if 'result' in globals() and result is not None:
    file_path = "procurement_report.html"

    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            html_content = f.read()

        html_content = html_content.strip()

        if html_content.startswith("```"):
            html_content = re.sub(r"^```(?:html)?\s*", "", html_content)
            html_content = re.sub(r"\s*```$", "", html_content)

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(html_content)

        display(HTML(html_content))

    else:
        print("HTML report file not found.")
        print("Raw output:")
        print(result.raw if hasattr(result, "raw") else result)

else:
    print("No result yet.")

Product Name,Score,Strengths,Weaknesses
Samsung Galaxy S21,9/10,"Meets all requirements, reliable brand, good camera, long battery life, and fits within the company budget",Slightly lower battery capacity compared to Google Pixel 6
Google Pixel 6,8.5/10,"Good camera, long battery life, and reliable brand","Higher price, barely fits within the company budget"
